cache the CodeSearchNet dataset. This dataset was created for the CodeSearchNet challenge and contains millions of functions from open source libraries on GitHub in several programming languages

In [ ]:
from datasets import load_dataset
raw_dataset = load_dataset("code_search_net", "python")

In [ ]:
raw_dataset["train"]

### Tranform the dataset  into list to list of text

In [ ]:
# Don't uncomment the following line unless your dataset is small!
# training_corpus = [raw_datasets["train"][i: i + 1000]["whole_func_string"] for i in range(0, len(raw_datasets["train"]), 1000)]

In [ ]:
# Python generator
# The problem with a generator object is that it can only be used once.
# training_corpus = (raw_dataset["train"][i:i +1000]["while_func_string"] for i in range(0, len(raw_dataset["train"]), 1000))

# to fix above issuse
# def get_training_corpus():
#     return (
#         raw_dataset["train"][i : i + 1000]["whole_func_string"]
#         for i in range(0, len(raw_dataset["train"]), 1000)
#     )
# order way:
def get_training_corpus():
  dataset = raw_dataset["train"]
  for start_idx in range(0, len(dataset), 1000):
    sample = dataset[start_idx: start_idx+1000]
    yield sample["whole_func_string"]

training_corpus = get_training_corpus()

# Training a new tokenizer

In [ ]:
!pip install -U transformers

In [ ]:
# using GPT2
from transformers import AutoTokenizer
old_tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [ ]:
tokenizer = old_tokenizer.train_new_from_iterator(training_corpus, 52000)

In [ ]:
example = '''def add_numbers(a, b):
    """Add the two numbers `a` and `b`."""
    return a + b'''

tokens = old_tokenizer.tokenize(example)
tokens

In [ ]:
tokens = tokenizer.tokenize(example)
tokens

In [ ]:
print(len(tokens))
print(len(old_tokenizer.tokenize(example)))

Saving the tokenizer

In [ ]:
tokenizer.save_pretrained("code-search-net-tokenizer")

In [ ]:
from huggingface_hub import notebook_login,  logout

notebook_login()
# logout()

In [ ]:
tokenizer.push_to_hub("code-search-net-tokenizer")

In [ ]:
# Replace "huggingface-course" below with your actual namespace to use your own tokenizer
tokenizer = AutoTokenizer.from_pretrained("huggingface-course/code-search-net-tokenizer")